In [ ]:
%pip install -q langchain langchain-aws langgraph boto3 python-dotenv

# Week 6 Monday — LangGraph Fundamentals

## Overview

Last week you learned to build agents with `create_agent()`. That approach gives the model full autonomy: it reads the user's message, picks tools, and decides its own execution path. For roughly 90% of agent use cases, that's exactly what you want.

But what about the other 10%? What happens when you need **guaranteed step ordering**, **branching logic**, or **explicit routing** that can't be left to prompt engineering? That's where **LangGraph** comes in. Instead of handing the model a task and hoping it figures out the workflow, you *define* the workflow as a **graph** — nodes for computation, edges for control flow, and typed state for data.

Today we go from `create_agent()` to `StateGraph`: from autonomous agents to **orchestrated workflows**.

## Learning Objectives

By the end of today you will be able to:

- Explain why and when `StateGraph` is preferred over `create_agent()`
- Build a `StateGraph` with typed state, nodes, and edges
- Define state using `TypedDict` and apply reducers with `Annotated`
- Route execution with conditional edges
- Design state schemas that keep data raw and lightweight

## Today's Roadmap

| Stage | Topic | Materials |
|-------|-------|-----------|
| **1** | LangGraph Fundamentals & StateGraph Basics | Readings 01–03 + `demo_01` |
| **2** | State Design Patterns | Reading 04 + `demo_02` |
| **3** | Building an Email Agent Workflow | Reading 05 + `demo_03` |

---

# Stage 1 — LangGraph Fundamentals & StateGraph Basics

## What Is LangGraph?

LangGraph is a library for building **stateful, graph-based workflows** with LLMs. While `create_agent()` gives you a powerful autonomous agent, LangGraph gives you **explicit control** over:

- **State** — what data flows through your system
- **Nodes** — individual processing steps (Python functions)
- **Edges** — how control flows between steps
- **Conditions** — decision points that route to different branches

Think of it this way:

| Single Agent (`create_agent`) | LangGraph (`StateGraph`) |
|-------------------------------|-------------------------|
| "Here's a task, figure it out" | "Here are the exact steps and decision points" |
| Autonomous | Orchestrated |

## The Philosophy: Thinking in Graphs

The LangGraph documentation introduces a crucial mental model: **think in graphs, not chains**.

A graph consists of:
- **Nodes** — where computation happens (your functions)
- **Edges** — paths connecting nodes (control flow)
- **State** — data that travels along edges between nodes

```
┌──────────────┐
│    START     │
└──────┬───────┘
       │
       ▼
┌──────────────┐
│   Classify   │
└──────┬───────┘
       │
   ┌───┴───┐
   ▼       ▼
┌──────┐ ┌──────┐
│ Path │ │ Path │
│  A   │ │  B   │
└──────┘ └──────┘
```

This visual representation **directly maps to code** — what you diagram IS what you build.

## `create_agent()` vs `StateGraph` — A Decision Guide

One of the most common questions: *"Should I use `create_agent()` or build a custom StateGraph?"*

The wrong choice leads to:
- **Over-engineering** — complex graphs for tasks an agent could handle
- **Under-engineering** — forcing autonomous agents into strict workflows they can't reliably execute

### The Core Trade-off: Autonomy vs Control

| `create_agent()` | `StateGraph` |
|------------------|-------------|
| Agent decides what to do | You define what happens |
| Flexible, adaptive | Predictable, deterministic |
| Relies on prompt engineering | Relies on code structure |
| Better for open-ended tasks | Better for structured workflows |
| Model chooses tool order | You define execution order |

### The 90/10 Rule

**Use `create_agent()` (~90% of cases)** when:
1. Open-ended interaction — user might ask anything
2. Tool selection varies per query
3. Step order doesn't matter
4. Conversation-driven interface

**Use `StateGraph` (~10% of cases)** when:
1. Steps must happen in a **guaranteed order**
2. You have **branching logic** with different processing paths
3. You need **human approval checkpoints**
4. You have a clear **multi-stage pipeline**

### The Decision Framework

```
1. Does the task require flexible tool usage based on user input?
   YES → create_agent()    NO → Continue

2. Must certain steps happen in a guaranteed order?
   YES → StateGraph        NO → create_agent()

3. Are there branching decisions with different processing paths?
   YES → StateGraph        NO → create_agent()

4. Is there a need for human approval at specific points?
   YES → StateGraph        NO → create_agent()

5. When in doubt → start with create_agent(), refactor later
```

## Graphs, Nodes, and Edges

These are the three primitives that every LangGraph workflow is built from. Master them and you can express any workflow.

### StateGraph — The Container

`StateGraph` is the blueprint that holds your entire workflow definition. It knows what type of state flows through, which nodes exist, and how they connect.

```python
from langgraph.graph import StateGraph
from typing import TypedDict

class MyState(TypedDict):
    input_text: str
    processed: bool

workflow = StateGraph(MyState)
```

### Nodes — Where Work Happens

Nodes are Python functions that:
1. **Receive** the current state
2. **Perform** some computation
3. **Return** updates to the state (only the keys you want to change)

```python
def my_node(state: MyState) -> dict:
    text = state["input_text"]
    return {"processed": True}  # LangGraph merges this into state
```

### Edges — Control Flow

Edges define how the workflow moves between nodes. LangGraph provides `START` and `END` constants:

```python
from langgraph.graph import START, END

workflow.add_edge(START, "step_one")       # Entry point
workflow.add_edge("step_one", "step_two")  # Linear flow
workflow.add_edge("step_two", END)          # Exit
```

### Conditional Edges — Branching Logic

Real workflows need branching. Use `add_conditional_edges()` with a **routing function** that examines state and returns the next node name:

```python
def route_decision(state):
    if state["priority"] == "high":
        return "urgent_handler"
    return "standard_handler"

workflow.add_conditional_edges(
    "classify",
    route_decision,
    {"urgent_handler": "urgent_handler", "standard_handler": "standard_handler"}
)
```

### Common Graph Patterns

**Linear Pipeline:**
```
START → step_1 → step_2 → step_3 → END
```

**Branch and Merge:**
```
START → classify ─┬─ path_a ─┬─ merge → END
                   └─ path_b ─┘
```

**Loop Back:**
```python
def should_continue(state):
    return "retry" if not state["success"] else "done"

workflow.add_conditional_edges("process", should_continue, {
    "retry": "process",  # Loop back to retry
    "done": END
})
```

### Compiling: Blueprint → Executable

After defining nodes and edges, **compile** the graph into a runnable application. Compilation validates that edges connect properly, checks for unreachable nodes, and creates an optimized execution plan:

```python
app = workflow.compile()
result = app.invoke({"input_text": "Hello", "processed": False})
```

## Demo 01 — StateGraph Basics

Let's build our first `StateGraph`. This demo creates a simple three-step processing pipeline that:

1. **Analyzes** input text (programmatic)
2. **Processes** the analyzed input (programmatic)
3. **Formats** the output using an LLM
4. **Summarizes** accumulated state

Key concepts to watch for:
- `TypedDict` for typed state
- `Annotated` with the `add` reducer to accumulate values
- Nodes returning partial state updates
- Linear edge flow: `START → analyze → process → format → summary → END`

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model

print("Environment loaded.")

In [ ]:
# PART 1 — State Definition

from typing import TypedDict, Annotated
from operator import add
from langgraph.graph import StateGraph, START, END

# TypedDict gives us simple type safety (no validation, fast).
# Compare with Pydantic BaseModel which adds validation but is slower.

class WorkflowState(TypedDict):
    input_text: str
    step_count: Annotated[int, add]           # Reducer: adds values together
    processing_log: Annotated[list[str], add]  # Reducer: appends to list
    final_output: str

print("WorkflowState defined.")
print("Fields:", list(WorkflowState.__annotations__.keys()))

Notice the `Annotated[int, add]` pattern above. When a node returns `{"step_count": 1}`, LangGraph doesn't *replace* the current value — it **adds** 1 to whatever is already there. Same for `processing_log`: returning a list appends it to the existing list. This is called a **reducer**.

In [ ]:
# PART 2 — Node Functions
# Each node receives the full state and returns ONLY the keys it updates.

def analyze_node(state: WorkflowState) -> dict:
    return {
        "step_count": 1,
        "processing_log": [f"Analyzed: '{state['input_text']}'"]
    }


def process_node(state: WorkflowState) -> dict:
    return {
        "step_count": 1,
        "processing_log": ["Processing complete"]
    }


def format_node(state: WorkflowState) -> dict:
    model = init_chat_model("bedrock:us.amazon.nova-2-lite-v1:0")
    response = model.invoke(
        f"Summarize this processing log in one sentence: {state['processing_log']}"
    )
    return {
        "step_count": 1,
        "processing_log": ["Formatted output"],
        "final_output": response.content
    }


def summary_node(state: WorkflowState) -> dict:
    return {
        "final_output": state["final_output"] + " | Final step count: " + str(state["step_count"])
    }

print("Node functions defined: analyze_node, process_node, format_node, summary_node")

In [ ]:
# PART 3 — Build the Graph and Run

workflow = StateGraph(WorkflowState)

workflow.add_node("analyze", analyze_node)
workflow.add_node("process", process_node)
workflow.add_node("format", format_node)
workflow.add_node("summary", summary_node)

workflow.add_edge(START, "analyze")
workflow.add_edge("analyze", "process")
workflow.add_edge("process", "format")
workflow.add_edge("format", "summary")
workflow.add_edge("summary", END)

app = workflow.compile()

result = app.invoke({
    "input_text": "Hello, LangGraph!",
    "step_count": 0,
    "processing_log": [],
    "final_output": ""
})

print(f"Steps completed: {result['step_count']}")
print(f"Processing log: {result['processing_log']}")
print(f"Output: {result['final_output']}")

### What Just Happened?

1. We defined **state** with `TypedDict` — a typed container for data flowing through the graph.
2. We used **reducers** (`Annotated[int, add]`) so that `step_count` accumulates across nodes instead of being overwritten.
3. Each **node** returned only the keys it changed — LangGraph merged those updates into the full state.
4. We connected nodes with **edges** to create a linear pipeline: `START → analyze → process → format → summary → END`.
5. We **compiled** the graph and **invoked** it like any other LangChain runnable.

This is the core pattern you'll use in every LangGraph workflow.

---

# Stage 2 — State Design Patterns

State is the lifeblood of every LangGraph workflow. It carries data between nodes, captures intermediate results, and determines routing decisions. **Poor state design** leads to bloated objects, missing data, and confusing node logic. **Good state design** makes workflows clear, efficient, and maintainable.

## State as TypedDict

LangGraph uses Python's `TypedDict` for state definitions. This gives you:
- **Type hints** for IDE autocomplete and error checking
- **Clear contracts** about what data flows through the workflow
- **Validation** at compile time

```python
from typing import TypedDict, Literal, Optional

class ConversationState(TypedDict):
    messages: list[BaseMessage]
    user_id: str
    intent: Literal["question", "complaint", "request", "unknown"]
    search_results: Optional[list[str]]
    needs_escalation: bool
```

## The Golden Rules of State Design

### Rule 1: Keep State Raw

Store raw data, not formatted text. Format when you need it.

```python
# BAD: Storing pre-formatted prompt
class BadState(TypedDict):
    formatted_prompt: str  # "You are helping John who is a Premium user..."

# GOOD: Store raw data, format on-demand in nodes
class GoodState(TypedDict):
    user_name: str
    user_tier: str
```

Why? Raw data is flexible — you can format it differently in different nodes. Pre-formatted strings are rigid.

### Rule 2: State Travels, Large Data Stays External

State is copied between nodes. Keep it lightweight.

| Include in State | Store Externally |
|------------------|------------------|
| Routing flags (`is_urgent`, `category`) | Large documents |
| Current step / phase | Vector embeddings |
| Accumulated chat messages | User profile data (reference by ID) |
| Intermediate results (summaries) | File contents |
| Error states | Images / media |

### Rule 3: Messages Are Special

For conversational workflows, `messages` is typically a list that **accumulates** rather than replaces. LangGraph handles this automatically using a reducer.

### Rule 4: Update Surgically

Each node should return **only** the fields it is responsible for. This makes debugging easier — you can trace which node set which fields.

## Common Pitfalls

**Pitfall 1 — Mutating state directly:**
```python
# WRONG:
def bad_node(state):
    state["items"].append("new_item")  # Mutates in place!
    return {}

# CORRECT:
def good_node(state):
    return {"items": state["items"] + ["new_item"]}
```

**Pitfall 2 — Returning the full state:**
```python
# WRONG:
def bad_node(state):
    return dict(state)  # Copies everything unnecessarily

# CORRECT:
def good_node(state):
    return {"processed": True}  # Only what changed
```

**Pitfall 3 — Missing required fields at invocation:**
```python
# Provide all fields (or use Optional / defaults)
result = app.invoke({"input": "hello", "output": ""})  # All fields present
```

## Demo 02 — State Design Patterns

This demo shows a conversational workflow with richer state design:

1. A **custom reducer** for message accumulation
2. **Literal types** for intent classification
3. **Metadata fields** separate from conversation data
4. The principle of keeping state raw and formatting on-demand

In [ ]:
# PART 1 — State with a Custom Reducer

from typing import TypedDict, Annotated, Literal
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage

def messages_reducer(existing: list[BaseMessage], new: list[BaseMessage]) -> list[BaseMessage]:
    return existing + new


class ConversationState(TypedDict):
    # Core conversation data with custom reducer
    messages: Annotated[list[BaseMessage], messages_reducer]

    # Session metadata (separate from messages)
    user_id: str
    session_started: bool

    # Processing flags
    intent: Literal["question", "action", "chitchat", "unknown"]
    requires_tool: bool

print("ConversationState defined with custom messages_reducer.")
print("Fields:", list(ConversationState.__annotations__.keys()))

The `messages_reducer` function receives the **existing** list and the **new** list and returns the combined result. When a node returns `{"messages": [AIMessage(...)]}`, LangGraph calls `messages_reducer(existing_messages, [AIMessage(...)])` instead of replacing the list outright.

In [ ]:
# PART 2 — Node Functions: Classify and Respond

def classify_intent(state: ConversationState) -> dict:
    model = init_chat_model("bedrock:us.amazon.nova-2-lite-v1:0")

    last_message = state["messages"][-1] if state["messages"] else ""

    classification = model.invoke(
        f"Classify this message as 'question', 'action', 'chitchat', or 'unknown': {last_message}"
    )

    content = classification.content.lower()
    if "question" in content:
        intent = "question"
    elif "action" in content:
        intent = "action"
    elif "chitchat" in content:
        intent = "chitchat"
    else:
        intent = "unknown"

    return {
        "intent": intent,
        "requires_tool": intent == "action",
        "session_started": True
    }


def respond(state: ConversationState) -> dict:
    model = init_chat_model("bedrock:us.amazon.nova-2-lite-v1:0")

    # Format prompt on-demand — state stays raw
    context = f"User intent: {state['intent']}. "
    if state["requires_tool"]:
        context += "This may require tool usage. "

    response = model.invoke([
        {"role": "system", "content": context + "Respond helpfully."},
        *state["messages"]
    ])

    return {
        "messages": [AIMessage(content=response.content)]
    }

print("Node functions defined: classify_intent, respond")

In [ ]:
# PART 3 — Build and Run

workflow2 = StateGraph(ConversationState)

workflow2.add_node("classify", classify_intent)
workflow2.add_node("respond", respond)

workflow2.add_edge(START, "classify")
workflow2.add_edge("classify", "respond")
workflow2.add_edge("respond", END)

conversation_app = workflow2.compile()

result = conversation_app.invoke({
    "messages": [HumanMessage(content="What's the weather like today?")],
    "user_id": "demo_user",
    "session_started": False,
    "intent": "unknown",
    "requires_tool": False
})

print(f"Intent: {result['intent']}")
print(f"Requires tool: {result['requires_tool']}")
print(f"Response: {result['messages'][-1].content}")

### What We Learned in Stage 2

- **Custom reducers** let you control how state fields accumulate across nodes.
- **Literal types** constrain classification fields to a fixed set of values.
- **Separating concerns** in state (messages vs metadata vs flags) keeps things clean.
- **Formatting on-demand** (the `context` string inside `respond`) keeps state raw and flexible.

---

# Stage 3 — Building an Email Agent Workflow

Now we bring everything together in a realistic workflow. We'll build an **email triage system** inspired by the LangGraph "Thinking in Graphs" documentation.

## Requirements

| Requirement | Implementation |
|-------------|----------------|
| Classify emails | Node with LLM call |
| Different paths per category | Conditional edges |
| Auto-respond to inquiries | Response generation node |
| Filter spam | Simple action node |
| Escalate urgent & complaints | Priority + escalation flag |

## Workflow Design

```
        ┌─────────────┐
        │   START     │
        └──────┬──────┘
               │
               ▼
        ┌─────────────┐
        │  classify   │
        └──────┬──────┘
               │
    ┌──────────┼──────────┬───────────┐
    ▼          ▼          ▼           ▼
┌────────┐ ┌────────┐ ┌─────────┐ ┌────────┐
│  spam  │ │ urgent │ │complaint│ │inquiry │
└────┬───┘ └────┬───┘ └────┬────┘ └────┬───┘
     │          │          │           │
     └──────────┴──────────┴───────────┘
                    │
                    ▼
             ┌─────────────┐
             │    END      │
             └─────────────┘
```

This is a classic **classify-then-route** pattern — one of the most common LangGraph architectures. The classification node runs an LLM, and a routing function examines the result to pick the next node.

## Demo 03 — Email Agent Workflow

This demo implements the full email triage workflow:

1. **State** captures email text, sender, classification, priority, response, and escalation flag
2. **Classification node** uses an LLM to categorize the email
3. **Routing function** examines the classification and picks a handler
4. **Handler nodes** set priority, draft responses, and decide on escalation
5. **Conditional edges** wire classification → handlers → END

In [ ]:
# PART 1 — State Definition

from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END

class EmailState(TypedDict):
    email_text: str
    sender: str
    classification: Literal["spam", "inquiry", "complaint", "urgent", "unknown"]
    priority: Literal["low", "medium", "high"]
    response: str
    escalate: bool

print("EmailState defined.")
print("Fields:", list(EmailState.__annotations__.keys()))

In [ ]:
# PART 2 — Classification Node and Routing Function

def classify_email(state: EmailState) -> dict:
    model = init_chat_model("bedrock:us.amazon.nova-2-lite-v1:0")

    response = model.invoke(
        f"""Classify this email into exactly one category: spam, inquiry, complaint, or urgent.

Email from {state['sender']}:
{state['email_text']}

Respond with only the category name."""
    )

    classification = response.content.strip().lower()
    if classification not in ("spam", "inquiry", "complaint", "urgent"):
        classification = "unknown"

    return {"classification": classification}


def route_by_classification(state: EmailState) -> str:
    classification = state["classification"]
    if classification == "spam":
        return "handle_spam"
    elif classification == "urgent":
        return "handle_urgent"
    elif classification == "complaint":
        return "handle_complaint"
    else:
        return "handle_inquiry"

print("classify_email and route_by_classification defined.")

In [ ]:
# PART 3 — Handler Nodes

def handle_spam(state: EmailState) -> dict:
    return {
        "priority": "low",
        "response": "[SPAM FILTERED - No response sent]",
        "escalate": False
    }


def handle_urgent(state: EmailState) -> dict:
    model = init_chat_model("bedrock:us.amazon.nova-2-lite-v1:0")
    response = model.invoke(
        f"""Draft a brief acknowledgment for this urgent email.
Be professional and indicate immediate attention.

Email: {state['email_text']}"""
    )
    return {
        "priority": "high",
        "response": response.content,
        "escalate": True
    }


def handle_complaint(state: EmailState) -> dict:
    model = init_chat_model("bedrock:us.amazon.nova-2-lite-v1:0")
    response = model.invoke(
        f"""Draft a professional response to this complaint.
Acknowledge the issue and express intent to resolve.

Complaint: {state['email_text']}"""
    )
    return {
        "priority": "high",
        "response": response.content,
        "escalate": True
    }


def handle_inquiry(state: EmailState) -> dict:
    model = init_chat_model("bedrock:us.amazon.nova-2-lite-v1:0")
    response = model.invoke(
        f"""Draft a helpful response to this inquiry.
Be friendly and informative.

Inquiry: {state['email_text']}"""
    )
    return {
        "priority": "medium",
        "response": response.content,
        "escalate": False
    }

print("Handler nodes defined: handle_spam, handle_urgent, handle_complaint, handle_inquiry")

In [ ]:
# PART 4 — Build the Graph with Conditional Edges

email_workflow = StateGraph(EmailState)

email_workflow.add_node("classify", classify_email)
email_workflow.add_node("handle_spam", handle_spam)
email_workflow.add_node("handle_urgent", handle_urgent)
email_workflow.add_node("handle_complaint", handle_complaint)
email_workflow.add_node("handle_inquiry", handle_inquiry)

email_workflow.add_edge(START, "classify")

email_workflow.add_conditional_edges(
    "classify",
    route_by_classification,
    {
        "handle_spam": "handle_spam",
        "handle_urgent": "handle_urgent",
        "handle_complaint": "handle_complaint",
        "handle_inquiry": "handle_inquiry"
    }
)

email_workflow.add_edge("handle_spam", END)
email_workflow.add_edge("handle_urgent", END)
email_workflow.add_edge("handle_complaint", END)
email_workflow.add_edge("handle_inquiry", END)

email_app = email_workflow.compile()

print("Email workflow compiled successfully!")
print("Flow: START → classify → [spam|urgent|complaint|inquiry] → END")

In [ ]:
# PART 5 — Test the Workflow with Different Emails

test_emails = [
    {
        "email_text": "URGENT: Server is down! Need immediate assistance!",
        "sender": "ops@company.com"
    },
    {
        "email_text": "Hi, I'd like to know more about your enterprise pricing options.",
        "sender": "customer@business.com"
    },
    {
        "email_text": "I've been waiting 3 weeks for my order. This is unacceptable!",
        "sender": "unhappy@customer.com"
    },
    {
        "email_text": "FREE MONEY! Click here NOW to claim your prize!!!",
        "sender": "spammer@suspicious.com"
    }
]

for email in test_emails:
    print("=" * 60)
    print(f"From: {email['sender']}")
    print(f"Email: {email['email_text'][:60]}...")

    result = email_app.invoke({
        "email_text": email["email_text"],
        "sender": email["sender"],
        "classification": "unknown",
        "priority": "low",
        "response": "",
        "escalate": False
    })

    print(f"Classification: {result['classification']}")
    print(f"Priority: {result['priority']}")
    print(f"Escalate: {result['escalate']}")
    print(f"Response: {result['response'][:120]}...")
    print()

### What We Built in Stage 3

The email workflow demonstrates the full LangGraph pattern:

1. **State** captures inputs, classification results, and outputs in a single `TypedDict`.
2. A **classification node** calls an LLM and writes the result back to state.
3. A **routing function** examines `state["classification"]` and returns the next node name — this is pure Python, no LLM needed.
4. **Conditional edges** (`add_conditional_edges`) wire the router to four handler nodes.
5. Each **handler** sets priority, drafts a response (some use an LLM, spam doesn't), and flags for escalation.
6. All handlers converge on `END`.

This is a pattern you'll reuse constantly: **classify → route → handle**.

---

# Key Takeaways

1. **LangGraph makes control flow explicit.** Where `create_agent()` gives the model autonomy, `StateGraph` gives *you* precise control over execution order, branching, and routing.

2. **The 90/10 rule.** Default to `create_agent()` for most applications. Reach for `StateGraph` when you need guaranteed ordering, branching logic, or human checkpoints.

3. **Think in graphs.** Nodes are functions. Edges are control flow. State carries data. What you diagram on a whiteboard maps directly to code.

4. **State design matters.** Use `TypedDict` for structure. Keep state raw (format on-demand). Use reducers (`Annotated`) for accumulating values. Update only what changed.

5. **Classify → Route → Handle.** The conditional-edge pattern is the workhorse of LangGraph workflows. A classification node writes to state, a pure-Python routing function picks the next node, and handler nodes do the work.

6. **Compile validates your graph.** Calling `workflow.compile()` checks for disconnected nodes, missing edges, and structural errors before you ever run the workflow.

## Up Next: Tuesday — Context Engineering

Tomorrow we move from *building* workflows to *controlling what each node sees*. Context engineering is the art of managing what information flows into each LLM call — system prompts, message history, retrieved context, and more. You'll learn to build workflows where every node gets exactly the context it needs, nothing more.